In [1]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv() #读取.env配置文件的值，进加载入系统变量，临时。


# ==================== 1. 定义工具 ====================
def get_weather(location: str) -> str:
    return f"Current weather in {location} is sunny"
# 求平方根，根号x
def square_root(x: float) -> float:
    return x ** 0.5

# 把函数放到dict中，方便根据函数名查找, dict[工具名] 得到函数
tools_by_name = {"get_weather": get_weather, "square_root": square_root}

# ==================== 2. 定义工具的LLM描述 ====================
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather of a location, the user should supply a location first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"]
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "square_root",
            "description": "Calculate the square root of a given number.",
            "parameters": {
                "type": "object",
                "properties": {
                    "x": {
                        "type": "number",
                        "description": "The number to calculate square root for",
                    }
                },
                "required": ["x"]
            },
        }
    },
]

# ==================== 3. LLM客户端 ====================
client = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com"
)

# ==================== 4. 定义发送消息的方法 ====================
def send_messages(messages):
    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=messages,
        tools=tools,# 发送消息时，携带工具信息
        extra_body={"thinking": {"type": "disabled"}}
    )
    return response.choices[0].message

# ==================== 5. 定义执行Tool调用的方法 ====================
def execute_tool(message, messages):
    """解析Tool信息，执行对应的函数，并将结果追加到消息列表"""
    tool_call = message.tool_calls[0]
    tool_id = tool_call.id#llM生成，作为这次工具调用的唯一标识
    tool_name = tool_call.function.name # 工具的名称，函数名
    #tool_call.function.arguments {"location": "深圳"}
    # {"location": "深圳"} 把json字符串转成dict
    tool_args = json.loads(tool_call.function.arguments)

    print(f"调用工具: {tool_name}, 参数: {tool_args}")

    # 根据函数名找到对应函数并执行
    # tools_by_name = {"get_weather": get_weather, "square_root": square_root}, 是一个字典
    # tools_by_name[tool_name]函数对象
    # name='get_weather' tools_by_name[get_weather]函数对象
    # 函数(参数)
    # **tool_args, dict的解构 要求：参数的keyword, 必须与字典的key相同
    result = tools_by_name[tool_name](**tool_args)
    print(f"工具结果: {result}")

    # 将AI消息和Tool结果追加到消息列表, 告诉我们，它需要调用哪个工具
    # 保留记忆
    messages.append(message)
    # 工具消息
    messages.append({
        "role": "tool",
        "tool_call_id": tool_id, # 模型返回的工具唯一标识
        "content": str(result) #工具调用结果
    })
    return messages

# ==================== 6. 主流程 ====================
# 准备用户消息
messages = [{"role": "user", "content": "深圳天气怎么样?"}]
print(f"User> {messages[0]['content']}")

# 第一次调用LLM，获取ToolCall
message = send_messages(messages)
# 大模型返回的要调用的工具信息
print(f"Tool> {message.tool_calls[0]}")

# 执行Tool并将结果传回LLM
messages = execute_tool(message, messages)

# 再次调用LLM，生成最终答案
message = send_messages(messages)
print(f"AI> {message.content}")

User> 深圳天气怎么样?
Tool> ChatCompletionMessageFunctionToolCall(id='call_00_lBa3AabuluEocROU2UMR3699', function=Function(arguments='{"location": "深圳"}', name='get_weather'), type='function', index=0)
调用工具: get_weather, 参数: {'location': '深圳'}
工具结果: Current weather in 深圳 is sunny
AI> 深圳现在天气是晴天 ☀️，适合出门活动。需要了解气温或未来几天的预报吗？
